# Airbnb Paris – Experiment 3: ConTextTab vs. TabPFN-Klassifikation
- Binäre Klassifikation als Outlier-Erkennung; Vergleich der beiden TFMs
- Zwei **Kontextvarianten** aus dem Train-Split mit **identischer Größe von 1500 Zeilen**: `natural` (natürliche Rate) und `balanced_1to4` (300/1200)
- Gleiche Kontextgröße in beiden Varianten → der Vergleich misst allein die Zusammensetzung, nicht die Kontextmenge
- 1500 ist das Maximum: Airbnb hat nur 360 Outlier im Train, ein 1:4-Kontext braucht ein Fünftel der Zeilen als Outlier
- Beide Varianten werden auf dem **vollen Test-Split in natürlicher Verteilung** ausgewertet → AUPRC bleibt vergleichbar
- ConTextTab: numerisch + Freitext (nativ); TabPFN: nur numerisch

In [ ]:
import os
import time
import torch  # vor sap_rpt_oss laden (TORCH_LIBRARY-Doppelregistrierung vermeiden)
import numpy as np
import pandas as pd
import mlflow
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve, auc
from tabpfn import TabPFNClassifier
from sap_rpt_oss import SAP_RPT_OSS_Classifier

SEED = int(os.environ.get("SEED", 1))
TEXT_COLS = ["name", "description", "neighborhood_overview", "host_about"]
N_CTX, N_OUT_BAL = 1500, 300  # 1:4 -> 300/1200; capped by Airbnb (only 360 outliers in train)
print("SEED", SEED)

## Daten & Split laden

In [ ]:
base = pd.read_csv(f"../../data/preprocessed/cleaned_airbnb_paris_seed{SEED}.csv").set_index("row_id")
texts = pd.read_csv("../../data/preprocessed/cleaned_text_airbnb_paris.csv", keep_default_na=False).set_index("row_id")
split = pd.read_csv(f"../../data/splits/split_airbnb_paris_seed{SEED}.csv").set_index("row_id")

df = base.join(texts[TEXT_COLS])
y = (1 - df["is_top_rating"]).astype(int)  # 1 = Outlier
X = df.drop(columns=["is_top_rating"])        # numerisch + Freitext
Xnum = X.drop(columns=TEXT_COLS)          # nur numerisch
print("Zeilen:", len(X), "| Verteilung:", y.value_counts().to_dict())

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("airbnb_paris_experiment_3")

## Kontextvarianten aus Train, gemeinsamer Testsatz
- Beide Kontexte haben exakt `N_CTX` Zeilen und unterscheiden sich **nur** in der Zusammensetzung
- `natural` bildet die Trainingsverteilung ab, `balanced_1to4` hebt die Outlier auf 1:4 an
- Der Testsatz ist in beiden Fällen identisch und unverändert

In [ ]:
tr_idx = split.index[split["split"] == "train"]
test_idx = split.index[split["split"] == "test"]

rng = np.random.RandomState(SEED)
out_tr = y.loc[tr_idx].index[y.loc[tr_idx] == 1].to_numpy()
in_tr = y.loc[tr_idx].index[y.loc[tr_idx] == 0].to_numpy()
rng.shuffle(out_tr)
rng.shuffle(in_tr)

n_out = round(N_CTX * y.loc[tr_idx].mean())
contexts = {
    "natural": np.concatenate([out_tr[:n_out], in_tr[:N_CTX - n_out]]),
    "balanced_1to4": np.concatenate([out_tr[:N_OUT_BAL], in_tr[:N_CTX - N_OUT_BAL]]),
}
print(f"Outlier im Train verfügbar: {len(out_tr)} (limitiert N_CTX auf {5 * len(out_tr)})")
for name, idx in contexts.items():
    assert len(idx) == N_CTX
    print(f"{name:14s} Kontext {len(idx)} (Outlier {int(y.loc[idx].sum())}) | "
          f"Test {len(test_idx)} (Outlier {int(y.loc[test_idx].sum())}, Rate {y.loc[test_idx].mean():.4f})")

## ConTextTab (numerisch + Freitext)
- SAP-rpt-1-oss, max_context_size=8192, bagging=8; Score = P(Outlier)

In [ ]:
y_test = y.loc[test_idx]
for setting, train_idx in contexts.items():
    y_train = y.loc[train_idx]
    outlier_col = sorted(y_train.unique().tolist()).index(1)

    clf = SAP_RPT_OSS_Classifier(max_context_size=8192, bagging=8)
    t0 = time.time()
    clf.fit(X.loc[train_idx], y_train)  # y als Series mit X-Index -> korrektes Alignment
    proba = clf.predict_proba(X.loc[test_idx])[:, outlier_col]
    pred = clf.predict(X.loc[test_idx])
    runtime = time.time() - t0

    prec, rec, _ = precision_recall_curve(y_test, proba)
    auprc, auroc = auc(rec, prec), roc_auc_score(y_test, proba)
    print(f"ConTextTab [{setting}] – AUPRC={auprc:.4f} AUC-ROC={auroc:.4f} t={runtime:.1f}s")
    print(classification_report(y_test, pred, target_names=["inlier", "outlier"], digits=4, zero_division=0))

    with mlflow.start_run(run_name=f"contexttab_{setting}"):
        mlflow.log_params({"context": setting, "n_train": len(train_idx), "n_test": len(test_idx),
                           "n_features": X.shape[1], "max_context_size": 8192, "bagging": 8, "seed": SEED})
        mlflow.log_metrics({"auprc": float(auprc), "auc_roc": float(auroc), "runtime_s": round(runtime, 2)})

## TabPFN (nur numerisch)
- Freitexte entfernt; Score = P(Outlier)

In [ ]:
for setting, train_idx in contexts.items():
    y_train = y.loc[train_idx]
    outlier_col = sorted(y_train.unique().tolist()).index(1)

    tab = TabPFNClassifier()
    t0 = time.time()
    tab.fit(Xnum.loc[train_idx].values, y_train.values)
    proba = tab.predict_proba(Xnum.loc[test_idx].values)[:, outlier_col]
    pred = tab.predict(Xnum.loc[test_idx].values)
    runtime = time.time() - t0

    prec, rec, _ = precision_recall_curve(y_test, proba)
    auprc, auroc = auc(rec, prec), roc_auc_score(y_test, proba)
    print(f"TabPFN [{setting}] – AUPRC={auprc:.4f} AUC-ROC={auroc:.4f} t={runtime:.1f}s")
    print(classification_report(y_test, pred, target_names=["inlier", "outlier"], digits=4, zero_division=0))

    with mlflow.start_run(run_name=f"tabpfn_classification_{setting}"):
        mlflow.log_params({"context": setting, "n_train": len(train_idx), "n_test": len(test_idx),
                           "n_features": Xnum.shape[1], "seed": SEED})
        mlflow.log_metrics({"auprc": float(auprc), "auc_roc": float(auroc), "runtime_s": round(runtime, 2)})